# Ch.6 — Instance Segmentation (Mask R-CNN)

> **Mission:** Detect and segment individual product instances (ProductionCV challenge).  
> **Dataset:** Synthetic retail shelf images with per-instance masks.  
> **Goal:** Achieve mAP@0.50 ≥ 85% and IoU ≥ 70% using Mask R-CNN.

## 1 · The Core Idea: Detect + Segment Each Instance

**Mask R-CNN** extends Faster R-CNN with a mask prediction branch:

1. **Faster R-CNN**: Detect objects (bounding boxes + class labels)
2. **Mask head**: Predict 28×28 binary mask per detected object
3. **RoIAlign**: Bilinear interpolation (no quantization errors)
4. **Multi-task loss**: $\mathcal{L}_{total} = \mathcal{L}_{cls} + \mathcal{L}_{bbox} + \mathcal{L}_{mask}$

**Key difference from semantic segmentation:**
- **Semantic**: Classify every pixel (entire image)
- **Instance**: Detect objects, then segment each individually

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dark theme
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'

## 2 · Dataset: Synthetic Retail Shelf with Instances

Create synthetic data with per-instance annotations:
- Each product gets a unique bounding box + mask
- Overlapping products (instance differentiation)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SyntheticInstanceDataset(Dataset):
    """Generate synthetic retail shelf images with instance-level annotations."""
    def __init__(self, num_samples=200, img_size=512):
        self.num_samples = num_samples
        self.img_size = img_size
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Create blank image
        image = np.ones((self.img_size, self.img_size, 3), dtype=np.float32) * 0.9
        
        # Generate instances (products)
        np.random.seed(idx)
        num_products = np.random.randint(5, 12)
        
        boxes = []
        labels = []
        masks = []
        
        for i in range(num_products):
            # Random product position and size
            x1 = np.random.randint(50, self.img_size - 150)
            y1 = np.random.randint(50, self.img_size - 150)
            w = np.random.randint(40, 100)
            h = np.random.randint(60, 140)
            x2 = min(x1 + w, self.img_size - 1)
            y2 = min(y1 + h, self.img_size - 1)
            
            # Product class (1=cereal, 2=milk, 3=soda)
            product_class = np.random.randint(1, 4)
            
            # Product color based on class
            if product_class == 1:  # Cereal (orange/red)
                color = np.array([0.9, 0.4, 0.2])
            elif product_class == 2:  # Milk (white/blue)
                color = np.array([0.8, 0.9, 1.0])
            else:  # Soda (red/green)
                color = np.array([0.9, 0.2, 0.2])
            
            # Draw product on image
            image[y1:y2, x1:x2] = color
            
            # Create instance mask
            mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
            mask[y1:y2, x1:x2] = 1
            
            boxes.append([x1, y1, x2, y2])
            labels.append(product_class)
            masks.append(mask)
        
        # Convert to tensors
        image = torch.from_numpy(image).permute(2, 0, 1)  # [3, H, W]
        boxes = torch.as_tensor(boxes, dtype=torch.float32)  # [N, 4]
        labels = torch.as_tensor(labels, dtype=torch.int64)  # [N]
        masks = torch.as_tensor(np.array(masks), dtype=torch.uint8)  # [N, H, W]
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'masks': masks,
            'image_id': torch.tensor([idx])
        }
        
        return image, target

# Create datasets
train_dataset = SyntheticInstanceDataset(num_samples=200, img_size=512)
val_dataset = SyntheticInstanceDataset(num_samples=50, img_size=512)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

# Visualize sample with instances
sample_img, sample_target = train_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original image with bounding boxes
axes[0].imshow(sample_img.permute(1, 2, 0))
for box, label in zip(sample_target['boxes'], sample_target['labels']):
    x1, y1, x2, y2 = box
    rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, 
                     edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-5, f'Class {label.item()}', 
                fontsize=10, color='lime', fontweight='bold')
axes[0].set_title('Instance Detection (Bounding Boxes)', fontsize=12)
axes[0].axis('off')

# Instance masks (different color per instance)
combined_mask = np.zeros((512, 512, 3))
colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 0, 1], [0, 1, 1]]
for i, mask in enumerate(sample_target['masks']):
    color = colors[i % len(colors)]
    combined_mask[mask.numpy() == 1] = color

axes[1].imshow(combined_mask)
axes[1].set_title(f'Instance Masks ({len(sample_target["masks"])} products)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('img/ch06-instance-data.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print(f"\nSample has {len(sample_target['boxes'])} products:")
for i, label in enumerate(sample_target['labels']):
    class_name = ['', 'Cereal', 'Milk', 'Soda'][label.item()]
    print(f"  Instance {i+1}: {class_name}")

## 3 · Load Pretrained Mask R-CNN

Use Torchvision's pretrained Mask R-CNN (COCO weights), then fine-tune for our 3 product classes.

In [ ]:
def get_mask_rcnn_model(num_classes):
    """
    Load pretrained Mask R-CNN with ResNet-50 backbone.
    Replace heads for custom number of classes.
    
    Args:
        num_classes: Total classes (including background)
    """
    # Load pretrained model (COCO weights)
    model = maskrcnn_resnet50_fpn(pretrained=True)
    
    # Replace box predictor (classification + box regression)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Replace mask predictor (instance segmentation)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    
    return model

# Create model (4 classes: background + 3 product types)
num_classes = 4  # background + cereal + milk + soda
model = get_mask_rcnn_model(num_classes).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"\nModel created:")
print(f"  Total parameters: {total_params:.2f}M")
print(f"  Trainable parameters: {trainable_params:.2f}M")

# Test forward pass
model.eval()
test_img = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    test_output = model([test_img])
print(f"\nTest inference output:")
print(f"  Boxes: {test_output[0]['boxes'].shape}")
print(f"  Labels: {test_output[0]['labels'].shape}")
print(f"  Scores: {test_output[0]['scores'].shape}")
print(f"  Masks: {test_output[0]['masks'].shape}")

## 4 · Training Loop

Train Mask R-CNN with multi-task loss:
- Classification loss (cross-entropy)
- Box regression loss (smooth L1)
- Mask loss (binary cross-entropy per RoI)

In [ ]:
# Optimizer (SGD with momentum, standard for Mask R-CNN)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler (step decay)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 10
train_losses = []

print("\nStarting training...\n")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass (returns loss dict during training)
        loss_dict = model(images, targets)
        
        # Sum all losses
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
    
    # Step learning rate
    lr_scheduler.step()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")

print(f"\n✅ Training complete! Final loss: {train_losses[-1]:.4f}")

## 5 · Visualize Training Progress

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(1, num_epochs+1), train_losses, color='#e74c3c', linewidth=2.5, marker='o', markersize=8)
ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Loss (cls + bbox + mask)', fontsize=12, fontweight='bold')
ax.set_title('Mask R-CNN Training Loss', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add text annotation for final loss
ax.text(num_epochs, train_losses[-1], f'  Final: {train_losses[-1]:.3f}', 
        fontsize=11, va='center', color='#2ecc71', fontweight='bold')

plt.tight_layout()
plt.savefig('img/ch06-training-loss.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## 6 · Inference and Visualization

Run trained Mask R-CNN on validation images, visualize detections + masks.

In [ ]:
# Run inference on validation samples
model.eval()

# Get 3 sample images
sample_images = []
sample_targets = []
for i in range(3):
    img, target = val_dataset[i]
    sample_images.append(img)
    sample_targets.append(target)

# Inference
with torch.no_grad():
    predictions = model([img.to(device) for img in sample_images])

# Visualize results
fig, axes = plt.subplots(3, 3, figsize=(16, 14))

class_names = ['Background', 'Cereal', 'Milk', 'Soda']
colors_boxes = ['red', 'lime', 'cyan', 'yellow']

for i in range(3):
    img = sample_images[i].permute(1, 2, 0).numpy()
    target = sample_targets[i]
    pred = predictions[i]
    
    # Filter predictions by confidence (> 0.7)
    keep = pred['scores'] > 0.7
    pred_boxes = pred['boxes'][keep].cpu()
    pred_labels = pred['labels'][keep].cpu()
    pred_scores = pred['scores'][keep].cpu()
    pred_masks = pred['masks'][keep].cpu()
    
    # 1. Original image
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Input Image', fontsize=12)
    axes[i, 0].axis('off')
    
    # 2. Ground truth
    axes[i, 1].imshow(img)
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box
        color = colors_boxes[label.item()]
        rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, 
                        edgecolor=color, facecolor='none')
        axes[i, 1].add_patch(rect)
    axes[i, 1].set_title(f'Ground Truth ({len(target["boxes"])} objects)', fontsize=12)
    axes[i, 1].axis('off')
    
    # 3. Predictions with masks
    axes[i, 2].imshow(img)
    
    # Overlay masks
    for box, label, score, mask in zip(pred_boxes, pred_labels, pred_scores, pred_masks):
        x1, y1, x2, y2 = box.int()
        color = colors_boxes[label.item()]
        
        # Draw bounding box
        rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, 
                        edgecolor=color, facecolor='none')
        axes[i, 2].add_patch(rect)
        
        # Draw label
        class_name = class_names[label.item()]
        axes[i, 2].text(x1, y1-5, f'{class_name} {score:.2f}', 
                       fontsize=9, color=color, fontweight='bold',
                       bbox=dict(facecolor='black', alpha=0.5, edgecolor='none'))
        
        # Overlay mask (semi-transparent)
        mask_resized = torch.nn.functional.interpolate(
            mask.unsqueeze(0), size=(y2-y1, x2-x1), mode='bilinear'
        ).squeeze(0).squeeze(0)
        mask_binary = (mask_resized > 0.5).numpy()
        
        # Create colored overlay
        overlay = np.zeros((512, 512, 4))
        if color == 'red':
            overlay[y1:y2, x1:x2, 0] = mask_binary
        elif color == 'lime':
            overlay[y1:y2, x1:x2, 1] = mask_binary
        elif color == 'cyan':
            overlay[y1:y2, x1:x2, 1:3] = mask_binary[:, :, np.newaxis]
        overlay[y1:y2, x1:x2, 3] = mask_binary * 0.4  # Alpha channel
        
        axes[i, 2].imshow(overlay)
    
    axes[i, 2].set_title(f'Predictions ({len(pred_boxes)} detected)', fontsize=12)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('img/ch06-instance-results.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n📊 Sample predictions summary:")
for i, pred in enumerate(predictions[:3]):
    keep = pred['scores'] > 0.7
    num_detected = keep.sum().item()
    num_gt = len(sample_targets[i]['boxes'])
    print(f"  Sample {i+1}: Detected {num_detected}/{num_gt} products")

## 7 · Calculate mAP and IoU Metrics

Evaluate instance segmentation quality using:
- **mAP@0.50**: Average Precision at IoU threshold 0.5
- **mIoU**: Mean Intersection over Union for masks

In [ ]:
def calculate_iou_instances(pred_mask, gt_mask):
    """Calculate IoU between two binary masks."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return intersection / union if union > 0 else 0.0

def evaluate_detection(predictions, targets, iou_threshold=0.5):
    """Simple mAP calculation at given IoU threshold."""
    all_precisions = []
    all_ious = []
    
    for pred, target in zip(predictions, targets):
        # Filter by confidence
        keep = pred['scores'] > 0.7
        pred_boxes = pred['boxes'][keep].cpu().numpy()
        pred_masks = pred['masks'][keep].cpu().numpy()
        gt_boxes = target['boxes'].cpu().numpy()
        gt_masks = target['masks'].cpu().numpy()
        
        if len(pred_boxes) == 0 or len(gt_boxes) == 0:
            continue
        
        # Match predictions to ground truth
        matched = set()
        for pred_idx in range(len(pred_boxes)):
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx in range(len(gt_boxes)):
                if gt_idx in matched:
                    continue
                
                # Calculate mask IoU
                pred_mask_bin = (pred_masks[pred_idx, 0] > 0.5)
                gt_mask_bin = (gt_masks[gt_idx] > 0.5)
                iou = calculate_iou_instances(pred_mask_bin, gt_mask_bin)
                
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            if best_iou >= iou_threshold and best_gt_idx != -1:
                matched.add(best_gt_idx)
                all_ious.append(best_iou)
        
        # Precision for this image
        precision = len(matched) / len(pred_boxes) if len(pred_boxes) > 0 else 0
        all_precisions.append(precision)
    
    mean_ap = np.mean(all_precisions) if all_precisions else 0
    mean_iou = np.mean(all_ious) if all_ious else 0
    
    return mean_ap, mean_iou

# Evaluate on full validation set
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        predictions = model(images)
        all_predictions.extend(predictions)
        all_targets.extend(targets)

# Calculate metrics
mAP_50, mIoU = evaluate_detection(all_predictions, all_targets, iou_threshold=0.5)

print("\n📊 Validation Metrics:")
print(f"  mAP@0.50: {mAP_50:.4f} (target ≥ 0.85)")
print(f"  mIoU: {mIoU:.4f} (target ≥ 0.70)")

# Constraint status
constraint_1 = "✅ Met" if mAP_50 >= 0.85 else f"⚡ In Progress ({mAP_50:.1%})" 
constraint_2 = "✅ Met" if mIoU >= 0.70 else f"⚡ In Progress ({mIoU:.1%})"

print(f"\n🎯 Constraint Status:")
print(f"  #1 Detection Accuracy: {constraint_1}")
print(f"  #2 Segmentation Quality: {constraint_2}")

## 8 · Per-Class Performance Analysis

In [ ]:
# Analyze per-class detection accuracy
class_counts = {1: 0, 2: 0, 3: 0}  # Cereal, Milk, Soda
class_detections = {1: 0, 2: 0, 3: 0}

for pred, target in zip(all_predictions, all_targets):
    # Count ground truth instances per class
    for label in target['labels']:
        class_counts[label.item()] += 1
    
    # Count detections per class (confidence > 0.7)
    keep = pred['scores'] > 0.7
    for label in pred['labels'][keep]:
        if label.item() in class_detections:
            class_detections[label.item()] += 1

# Plot per-class stats
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_labels = ['Cereal', 'Milk', 'Soda']
gt_counts = [class_counts[i+1] for i in range(3)]
det_counts = [class_detections[i+1] for i in range(3)]

# Count comparison
x = np.arange(3)
width = 0.35
axes[0].bar(x - width/2, gt_counts, width, label='Ground Truth', color='#3498db', edgecolor='white')
axes[0].bar(x + width/2, det_counts, width, label='Detected (>0.7)', color='#2ecc71', edgecolor='white')
axes[0].set_xlabel('Product Class', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12, fontweight='bold')
axes[0].set_title('Per-Class Instance Counts', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(class_labels)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Detection recall
recalls = [det_counts[i] / gt_counts[i] if gt_counts[i] > 0 else 0 for i in range(3)]
colors = ['#2ecc71' if r >= 0.85 else '#f39c12' for r in recalls]
bars = axes[1].bar(class_labels, recalls, color=colors, edgecolor='white', linewidth=2)

# Add value labels
for bar, recall in zip(bars, recalls):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{recall:.2%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].set_xlabel('Product Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Recall', fontsize=12, fontweight='bold')
axes[1].set_title('Per-Class Detection Recall', fontsize=14, fontweight='bold')
axes[1].axhline(y=0.85, color='#e74c3c', linestyle='--', linewidth=2, label='Target (85%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('img/ch06-per-class-performance.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n📊 Per-Class Recall:")
for label, recall in zip(class_labels, recalls):
    status = "✅" if recall >= 0.85 else "⚡"
    print(f"  {status} {label}: {recall:.2%}")

## 9 · Exercises

**Exercise 1:** Modify RoI score threshold (0.5, 0.7, 0.9) — observe precision/recall trade-off.  
**Exercise 2:** Add more product classes (chips, cookies) — does mAP degrade?  
**Exercise 3:** Measure inference time per image — how does it scale with # detected objects?

In [ ]:
# Exercise 1: Threshold sensitivity
# TODO: Test thresholds [0.5, 0.6, 0.7, 0.8, 0.9]
# for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
#     filter predictions by threshold
#     calculate precision and recall
#     plot precision-recall curve

# Exercise 2: Inference time profiling
# TODO: Measure time per forward pass
# import time
# start = time.time()
# with torch.no_grad():
#     predictions = model([test_image.to(device)])
# elapsed = time.time() - start
# print(f"Inference time: {elapsed*1000:.2f} ms")

# Exercise 3: Model size analysis
# TODO: Save model and check file size
# torch.save(model.state_dict(), 'mask_rcnn_weights.pth')
# import os
# size_mb = os.path.getsize('mask_rcnn_weights.pth') / (1024 * 1024)
# print(f"Model size: {size_mb:.2f} MB (target < 100 MB)")

print("💡 Exercises for further exploration:")
print("  1. Experiment with RoI score thresholds (precision vs recall)")
print("  2. Profile inference time (relate to constraint #3: <50ms)")
print("  3. Measure model size (constraint #4: <100 MB)")